# DQMBot — Batch Image Query Driver

**Image layout expected:**
```
images/
    <plotName>/
        <plotName>_run<XXXXXX>.png
```

**Output layout (with run_id — preserves previous runs):**
```
results/
    <run_id>/
        <plotName>/
            <plotName>_<model>_run<XXXXXX>.txt
        summary_<run_id>.csv
```

**Output layout (no run_id — overwrites):**
```
results/
    <plotName>/
        <plotName>_<model>_run<XXXXXX>.txt
    summary.csv
```

In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from owui_client import (
    list_models, query,
    batch_query_images, _collect_images, resolve_output_dir, resolve_output_file,
    retrieve_and_inspect,
)

print('owui_client loaded OK')

In [ ]:
# ── Dependencies (run once, then restart kernel) ───────────────────────────────
# !pip install rank-bm25 langchain-huggingface sentence-transformers --quiet

In [ ]:
# ── Discover available models ──────────────────────────────────────────────────
print('=== Models ===')
for m in list_models():
    print(' ', m)

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

IMAGE_ROOT  = Path('images')
OUTPUT_ROOT = Path('results')

# Set a string to preserve previous runs alongside this one.
# Leave as None to overwrite.
RUN_ID = 'baseline'
# RUN_ID = None

MODELS = [
 'qwen2.5vl:latest',            #7b
 'qwen2.5vl:32b',               #32b
 'qwen3-vl:latest',             #8b
 'litellm-ow.qwen/qwen3.6',     #35b
 'gemma3:latest',               #4b
 'litellm-ow.google/gemma4-31b',#31b
]

RAG_BACKEND = 'local'                          # 'local' or 'owui'
CSV_PATH    = Path('document_chunks.csv')      # used when RAG_BACKEND='local'

SYSTEM_PROMPT = (
 """\
You are an assistant to shifters of the CMS experiment during detector operations.
You are tasked to judge if a input plot is good or bad.

In your output, make 4 sections:
 - Quote the relevant section of instructions for the input plot
 - Describe the input plot
 - Compare input plot to the instruction
 - Decide if the plot is good or bad\
"""
)

PROMPT = (
    ''
)

DELAY = 1.5
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
# ── Debug: inspect what RAG retrieves for the query ──────────────────────────
debug_query = "ECal TP ET-weighted Occupancy"
hits = retrieve_and_inspect(debug_query, csv_path=CSV_PATH, top_k=5, method="hybrid")

print(f'Query: "{debug_query}"\n')
for h in hits:
    print(f"  #{h['rank']}  score={h['score']:.4f}  doc={h['document_id']}  "
          f"chunk={h['chunk_index']}  len={h['text_length']}")
    print(f"       source: {h['source']}")
    print(f"       preview: {h['text_preview'][:120]}...")
    print()

In [ ]:
# ── Sanity check: show what will be processed and where it will land ──────────
pairs = _collect_images(IMAGE_ROOT, ('.png', '.jpg', '.jpeg', '.webp'))
plot_names = sorted(set(p for p, _ in pairs))

print(f'Plots found  : {len(plot_names)}')
for pn in plot_names:
    imgs = [img for p, img in pairs if p == pn]
    print(f'  {pn}/  ({len(imgs)} images)')
    for img in imgs:
        print(f'    {img.name}')

print(f'\nModels       : {len(MODELS)}')
for m in MODELS:
    print(f'  {m}')

print(f'\nRun ID       : {RUN_ID or "(none — overwrite mode)"}')
print(f'Total queries: {len(pairs) * len(MODELS)}')

print('\nExample output paths:')
for model in MODELS:
    plot_name, img = pairs[0]
    d = resolve_output_dir(OUTPUT_ROOT, plot_name, RUN_ID)
    f = resolve_output_file(d, img, model)
    print(f'  {f}')

In [ ]:
# ── Smoke test: one image, first model ───────────────────────────────────────
if pairs:
    plot_name, img = pairs[0]
    test = query(
        PROMPT,
        model=MODELS[2],
        system=SYSTEM_PROMPT,
        image_path=img,
        rag_backend=RAG_BACKEND,
        csv_path=CSV_PATH,
    )
    print(f"Plot    : {plot_name}")
    print(f"Model   : {test['model_used']}")
    print(f"Image   : {test['image']}")
    print(f"Latency : {test['latency_s']}s")
    print(f"Error   : {test['error']}")
    print()
    print(test['response'])

In [ ]:
# ── Full batch ────────────────────────────────────────────────────────────────
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

results = batch_query_images(
    PROMPT,
    image_root=IMAGE_ROOT,
    models=MODELS,
    output_root=OUTPUT_ROOT,
    run_id=RUN_ID,
    system=SYSTEM_PROMPT,
    rag_backend=RAG_BACKEND,
    csv_path=CSV_PATH,
    delay=DELAY,
    verbose=True,
)

print(f'\nDone. {len(results)} queries completed.')

In [ ]:
# ── Retry errors ──────────────────────────────────────────────────────────────
failed = [
    (r['model_used'], r['image'])
    for r in results if r['error'] is not None
]

if not failed:
    print('No errors in results — nothing to retry.')
else:
    print(f'Retrying {len(failed)} failed quer{"y" if len(failed)==1 else "ies"}...')
    retry_results = []

    for i, (model, image_str) in enumerate(failed):
        image_path = Path(image_str)
        plot_name  = image_path.parent.name
        print(f'  [{i+1}/{len(failed)}] model={model}  image={image_path.name} ...', end=' ', flush=True)

        result = query(
            PROMPT,
            model=model,
            system=SYSTEM_PROMPT,
            image_path=image_path,
            rag_backend=RAG_BACKEND,
            csv_path=CSV_PATH,
        )
        result['plot_name'] = plot_name
        retry_results.append(result)

        out_dir  = resolve_output_dir(OUTPUT_ROOT, plot_name, RUN_ID)
        out_dir.mkdir(parents=True, exist_ok=True)
        out_file = resolve_output_file(out_dir, image_path, model)
        with open(out_file, 'w') as f:
            f.write(f"Model:    {result['model_used']}\n")
            f.write(f"Plot:     {plot_name}\n")
            f.write(f"Image:    {result['image']}\n")
            f.write(f"Run ID:   {RUN_ID or '(overwrite)'}\n")
            f.write(f"Latency:  {result['latency_s']}s\n")
            f.write(f"Prompt:   {result['prompt']}\n")
            f.write('-' * 60 + '\n')
            if result['error']:
                f.write(f"ERROR: {result['error']}\n")
            else:
                f.write(result['response'] + '\n')

        status = 'ERROR' if result['error'] else f"{result['latency_s']}s → {out_file}"
        print(status)
        time.sleep(DELAY)

    retry_index = {(r['model_used'], r['image']): r for r in retry_results}
    results = [
        retry_index.get((r['model_used'], r['image']), r)
        for r in results
    ]

    still_failing = sum(1 for r in retry_results if r['error'])
    print(f'\nDone. {len(retry_results) - still_failing}/{len(retry_results)} recovered.')
    if still_failing:
        print('Still failing:')
        for r in retry_results:
            if r['error']:
                print(f"  {r['model_used']}  {Path(r['image']).name}  → {r['error']}")

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
df = pd.DataFrame(results)
df['image_name'] = df['image'].apply(lambda p: Path(p).name if p else None)

errors = df[df['error'].notna()]
if not errors.empty:
    print(f'WARNING: {len(errors)} failed queries:')
    display(errors[['plot_name', 'model_used', 'image_name', 'error']])
else:
    print('All queries succeeded.')

print()
display(
    df.groupby(['plot_name', 'model_used'])['latency_s']
      .agg(['count', 'mean', 'min', 'max'])
      .round(2)
      .rename(columns={'count': 'n', 'mean': 'avg_s', 'min': 'min_s', 'max': 'max_s'})
)

In [ ]:
# ── Save CSV next to the run's output folder ──────────────────────────────────
if RUN_ID:
    csv_path = OUTPUT_ROOT / RUN_ID / f'summary_{RUN_ID}.csv'
else:
    csv_path = OUTPUT_ROOT / 'summary.csv'

df.to_csv(csv_path, index=False)
print(f'Saved: {csv_path}')

# Show result tree
print()
root = OUTPUT_ROOT / RUN_ID if RUN_ID else OUTPUT_ROOT
for item in sorted(root.iterdir()):
    if item.is_dir():
        txts = list(item.glob('*.txt'))
        print(f'  {item.name}/  ({len(txts)} files)')
        for t in sorted(txts):
            print(f'    {t.name}')
    elif item.suffix == '.csv':
        print(f'  {item.name}')

In [ ]:
# ── Side-by-side comparison: one image across all models ─────────────────────
COMPARE_PLOT  = plot_names[0]
COMPARE_IMAGE = pairs[0][1].name

subset = df[(df['plot_name'] == COMPARE_PLOT) & (df['image_name'] == COMPARE_IMAGE)]
for _, row in subset.iterrows():
    print('=' * 72)
    print(f"Model   : {row['model_used']}")
    print(f"Latency : {row['latency_s']}s")
    print()
    print(row['error'] and f"ERROR: {row['error']}" or row['response'])
    print()